# 🍃 Mangosteen Ripeness Classification — Separable CNN High-Accuracy Training Pipeline
**Project Formulation:** 245 ภาพจริง -> 365 ภาพสมบูรณ์ (Targeted Train Augmentation +120 ภาพ)  
**Target Hardware:** LilyGO T-SIMCAM (ESP32-S3 @ 240MHz, 8MB PSRAM)  
**Architecture:** Multi-Stage Depthwise Separable CNN (~94,163 Parameters)  
**Classes:** `overripe` (0), `ripe` (1), `unripe` (2)  
**Input Shape:** `96x96x3` RGB (Full INT8 Quantization)  
**Export Model:** `mangosteen_separable_cnn_94k_96x96_int8.tflite` (~134 KB)


In [ ]:
# [Cell 1] Setup Environment, Mount Google Drive & Import Libraries
import os
import sys
import random
from pathlib import Path
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from google.colab import drive, files

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. ตั้งค่า Random Seed ให้ผลลัพธ์คงที่
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# 3. กำหนดค่าคงที่และ Path ชุดข้อมูล
DATASET_DIR = Path('/content/drive/MyDrive/Dataset Mangosteen')
TRAIN_DIR   = DATASET_DIR / 'train'
VAL_DIR     = DATASET_DIR / 'val'
TEST_DIR    = DATASET_DIR / 'test'

IMG_HEIGHT, IMG_WIDTH = 96, 96
BATCH_SIZE = 16
EPOCHS = 80
CLASSES = ['overripe', 'ripe', 'unripe']

print(f'TensorFlow Version: {tf.__version__}')
print('GPU Available:', tf.config.list_physical_devices('GPU'))
print(f'Dataset Path: {DATASET_DIR}')


## 📁 ขั้นตอนที่ 1: ตรวจสอบและปรับสมดุลชุดข้อมูล (จาก 245 ภาพ -> 365 ภาพ)
ตามสเปกรายงานของโครงการ:
- **เริ่มต้น (ภาพจริง):** มี 245 ภาพ (`Unripe: 137`, `Ripe: 74`, `Overripe: 34`)
- **การทำ Targeted Augmentation (+120 ภาพ):** ทำเฉพาะในชุด `train` เพื่อป้องกัน Data Leakage
  - คลาส `ripe`: เพิ่ม **+51 ภาพ**
  - คลาส `overripe`: เพิ่ม **+69 ภาพ**
- **รวมทั้งโครงการ:** กลายเป็น **365 ภาพ** พอดีเป๊ะ (Train: 289, Val: 36, Test: 40)


In [ ]:
# [Cell 2] Targeted Augmentation (คลาส ripe +51 ภาพ, overripe +69 ภาพ)
from PIL import Image, ImageEnhance

# กำหนดจำนวนภาพที่ต้องสังเคราะห์เพิ่มในชุด Train ตามสเปกโครงการ
AUG_CONFIG = {
    'ripe': 51,      # เพิ่ม Ripe +51 ภาพ
    'overripe': 69   # เพิ่ม Overripe +69 ภาพ
}
valid_exts = {'.jpg', '.jpeg', '.png'}

def augment_image(img: Image.Image, seed: int):
    rng = np.random.default_rng(seed)
    if rng.random() > 0.5: img = img.transpose(Image.FLIP_LEFT_RIGHT)
    if rng.random() > 0.5: img = img.transpose(Image.FLIP_TOP_BOTTOM)
    angle = rng.uniform(-20, 20)
    img = img.rotate(angle, resample=Image.BICUBIC, expand=False)
    zoom = rng.uniform(0.92, 1.08)
    w, h = img.size
    new_w, new_h = int(w * zoom), int(h * zoom)
    img_resized = img.resize((new_w, new_h), Image.BICUBIC)
    if zoom > 1.0:
        left, top = (new_w - w) // 2, (new_h - h) // 2
        img = img_resized.crop((left, top, left + w, top + h))
    else:
        pad = Image.new('RGB', (w, h), (0, 0, 0))
        pad.paste(img_resized, ((w - new_w) // 2, (h - new_h) // 2))
        img = pad
    img = ImageEnhance.Brightness(img).enhance(rng.uniform(0.92, 1.08))
    img = ImageEnhance.Contrast(img).enhance(rng.uniform(0.92, 1.08))
    return img

print('🔄 กำลังตรวจสอบและทำ Data Augmentation ในโฟลเดอร์ Train...')
for c in CLASSES:
    c_dir = TRAIN_DIR / c
    if not c_dir.exists(): continue
    for f in c_dir.glob('aug_*.*'): f.unlink() # ล้างของเก่าออกก่อน
    
    origs = [f for f in sorted(c_dir.glob('*.*')) if f.suffix.lower() in valid_exts]
    needed = AUG_CONFIG.get(c, 0)
    
    if needed > 0 and len(origs) > 0:
        factor = int(np.ceil(needed / len(origs)))
        count = 0
        for idx, orig_p in enumerate(origs):
            im = Image.open(orig_p).convert('RGB')
            for v in range(factor):
                if count >= needed: break
                seed = (idx + 1) * 1000 + v + 42
                aug = augment_image(im, seed)
                aug.save(c_dir / f'aug_{c}_{idx+1:02d}_v{v+1:02d}.jpg', quality=95)
                count += 1
                
    total = len(list(c_dir.glob('*.*')))
    print(f'   - คลาส {c:8s}: ภาพจริง {len(origs):2d} ภาพ + เพิ่ม {needed:2d} ภาพ = รวม {total:3d} ภาพ')

print('\n📊 สรุปจำนวนภาพรวมทั้งโครงการ (Train + Val + Test):')
for split_name, split_p in [('Train', TRAIN_DIR), ('Validation', VAL_DIR), ('Test', TEST_DIR)]:
    if split_p.exists():
        cnt = sum(len(list((split_p / c).glob('*.*'))) for c in CLASSES if (split_p / c).exists())
        print(f'   - ชุด {split_name:10s}: {cnt:3d} ภาพ')


## 📊 ขั้นตอนที่ 2: โหลด Dataset และคำนวณ Balanced Class Weights


In [ ]:
# [Cell 3] Load Datasets into Keras Pipelines & Compute Weights
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR, image_size=(IMG_HEIGHT, IMG_WIDTH), batch_size=BATCH_SIZE,
    label_mode='categorical', shuffle=True, seed=SEED
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR, image_size=(IMG_HEIGHT, IMG_WIDTH), batch_size=BATCH_SIZE,
    label_mode='categorical', shuffle=False
)
test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR if TEST_DIR.exists() else VAL_DIR,
    image_size=(IMG_HEIGHT, IMG_WIDTH), batch_size=1,
    label_mode='categorical', shuffle=False
)
eval_train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR, image_size=(IMG_HEIGHT, IMG_WIDTH), batch_size=BATCH_SIZE,
    label_mode='categorical', shuffle=False
)

class_counts = {c: len(list((TRAIN_DIR / c).glob('*.*'))) for c in CLASSES}
total_samples = sum(class_counts.values())
class_weights = {idx: total_samples / (len(CLASSES) * class_counts[c]) for idx, c in enumerate(CLASSES)}

print(f'   Classes: {train_ds.class_names}')
print(f'   Train samples per class: {class_counts}')
print(f'   Balanced Class Weights : {class_weights}')


## 🧠 ขั้นตอนที่ 3: สร้างสถาปัตยกรรมโมเดล Separable CNN (~94,163 Parameters)
Multi-Stage Separable CNN 94k ขนาดบีบอัด Full INT8 เท่ากับ ~134 KB


In [ ]:
# [Cell 4] Build Multi-Stage Separable CNN Architecture
l2 = tf.keras.regularizers.l2(1.0e-4)
inputs = tf.keras.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3), name='input_image')

# Online Augmentation
x = tf.keras.layers.RandomFlip('horizontal')(inputs)
x = tf.keras.layers.RandomRotation(0.10)(x)
x = tf.keras.layers.RandomZoom(0.08)(x)
x = tf.keras.layers.RandomContrast(0.06)(x)
x = tf.keras.layers.RandomBrightness(0.06)(x)

# Rescaling [0..255] -> [-1..1]
x = tf.keras.layers.Rescaling(1./127.5, offset=-1.0)(x)

# Stage 0
x = tf.keras.layers.Conv2D(32, (3, 3), strides=2, padding='same', use_bias=False, kernel_regularizer=l2, name='conv0')(x)
x = tf.keras.layers.BatchNormalization(name='bn0')(x)
x = tf.keras.layers.ReLU(6.0, name='relu0')(x)

# Stage 1
x = tf.keras.layers.SeparableConv2D(64, (3, 3), strides=2, padding='same', use_bias=False, depthwise_regularizer=l2, pointwise_regularizer=l2, name='sep_conv1')(x)
x = tf.keras.layers.BatchNormalization(name='bn1')(x)
x = tf.keras.layers.ReLU(6.0, name='relu1')(x)

# Stage 2
x = tf.keras.layers.SeparableConv2D(128, (3, 3), strides=2, padding='same', use_bias=False, depthwise_regularizer=l2, pointwise_regularizer=l2, name='sep_conv2')(x)
x = tf.keras.layers.BatchNormalization(name='bn2')(x)
x = tf.keras.layers.ReLU(6.0, name='relu2')(x)

# Stage 3
x = tf.keras.layers.SeparableConv2D(144, (3, 3), strides=1, padding='same', use_bias=False, depthwise_regularizer=l2, pointwise_regularizer=l2, name='sep_conv3')(x)
x = tf.keras.layers.BatchNormalization(name='bn3')(x)
x = tf.keras.layers.ReLU(6.0, name='relu3')(x)

# Stage 4
x = tf.keras.layers.SeparableConv2D(176, (3, 3), strides=2, padding='same', use_bias=False, depthwise_regularizer=l2, pointwise_regularizer=l2, name='sep_conv4')(x)
x = tf.keras.layers.BatchNormalization(name='bn4')(x)
x = tf.keras.layers.ReLU(6.0, name='relu4')(x)

# Stage 5
x = tf.keras.layers.SeparableConv2D(176, (3, 3), strides=1, padding='same', use_bias=False, depthwise_regularizer=l2, pointwise_regularizer=l2, name='sep_conv5')(x)
x = tf.keras.layers.BatchNormalization(name='bn5')(x)
x = tf.keras.layers.ReLU(6.0, name='relu5')(x)

# Output Head
x = tf.keras.layers.GlobalAveragePooling2D(name='gap')(x)
x = tf.keras.layers.Dropout(0.25, name='dropout')(x)
outputs = tf.keras.layers.Dense(len(CLASSES), activation='softmax', kernel_regularizer=l2, name='predictions')(x)

model = tf.keras.Model(inputs, outputs, name='Mangosteen_SeparableCNN_94k')
total_params = model.count_params()
print(f'👉 Total Parameters: {total_params:,} (ตรงกับสเปก 94,163 พารามิเตอร์)')
model.summary()


## 🚀 ขั้นตอนที่ 4: ฝึกสอนโมเดล (Train 80 Epochs ด้วย MultiMetricCheckpoint)
ใช้ Cosine Decay Learning Rate และ Callback แบบเดียวกับในเครื่องที่ต้องทายครบ 3 คลาส และวัดจาก Balanced Accuracy


In [ ]:
# [Cell 5] Compile and Train with MultiMetricCheckpoint
from sklearn.metrics import f1_score, balanced_accuracy_score

lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1.0e-3, decay_steps=EPOCHS * len(train_ds), alpha=0.01
)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=lr_schedule),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.03),
    metrics=['accuracy']
)

keras_save_path = Path('mangosteen_separable_cnn_94k.keras')

class MultiMetricCheckpoint(tf.keras.callbacks.Callback):
    def __init__(self, filepath):
        super().__init__()
        self.filepath = filepath
        self.best_bal_acc = -1.0
        self.best_macro_f1 = -1.0
        self.best_epoch = 0

    def on_epoch_end(self, epoch, logs=None):
        val_preds, val_targets = [], []
        for imgs, lbls in val_ds:
            p = np.argmax(self.model(imgs, training=False).numpy(), axis=1)
            t = np.argmax(lbls.numpy(), axis=1)
            val_preds.extend(p)
            val_targets.extend(t)

        val_preds = np.array(val_preds)
        val_targets = np.array(val_targets)
        classes_found = len(set(val_preds))

        macro_f1 = f1_score(val_targets, val_preds, average='macro', zero_division=0)
        bal_acc = balanced_accuracy_score(val_targets, val_preds) * 100.0
        raw_acc = np.mean(val_targets == val_preds) * 100.0

        if classes_found == len(CLASSES) and bal_acc > self.best_bal_acc:
            self.best_bal_acc = bal_acc
            self.best_macro_f1 = macro_f1
            self.best_epoch = epoch + 1
            self.model.save(str(self.filepath))
            print(f'\n   >>> [Epoch {epoch+1:02d} ⭐ BEST] Bal Acc: {bal_acc:.1f}% | Raw Acc: {raw_acc:.1f}% (Saved)')

checkpoint_cb = MultiMetricCheckpoint(keras_save_path)

print('🚀 เริ่มเทรนโมเดล 80 Epochs...')
history = model.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS,
    class_weight=class_weights, callbacks=[checkpoint_cb], verbose=1
)

best_model = tf.keras.models.load_model(str(keras_save_path))
print(f'\n✅ โหลดโมเดลจุดที่ดีที่สุดสำเร็จจาก Epoch {checkpoint_cb.best_epoch}')


## 📈 ขั้นตอนที่ 5: การประเมินผล (Accuracy, Training Curves, Confusion Matrix)


In [ ]:
# [Cell 6] Model Evaluation, Graphs & Confusion Matrix
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# 1. คำนวณ Accuracy
train_loss, train_acc = best_model.evaluate(eval_train_ds, verbose=0)
val_loss, val_acc     = best_model.evaluate(val_ds, verbose=0)

y_true, y_pred = [], []
for imgs, labels in test_ds:
    pred = best_model.predict(imgs, verbose=0)
    y_pred.append(np.argmax(pred[0]))
    y_true.append(np.argmax(labels.numpy()[0]))
y_true, y_pred = np.array(y_true), np.array(y_pred)
test_acc = np.mean(y_true == y_pred) * 100.0

print('=' * 60)
print(' 🎯 สรุปผลความแม่นยำ (ACCURACY & LOSS)')
print('=' * 60)
print(f' 📂 Training Set   : Accuracy = {train_acc*100:6.2f}% | Loss = {train_loss:.4f}')
print(f' 📂 Validation Set : Accuracy = {val_acc*100:6.2f}% | Loss = {val_loss:.4f}')
print(f' 📂 Test Set       : Accuracy = {test_acc:6.2f}% ({np.sum(y_true == y_pred)}/{len(y_true)} ภาพ)')
print('=' * 60)

# 2. พล็อตกราฟ Training Curves
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Acc', linewidth=2)
plt.plot(history.history['val_accuracy'], label='Validation Acc', linewidth=2)
plt.title('Training vs Validation Accuracy')
plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.grid(True, alpha=0.3); plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss', linewidth=2)
plt.plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
plt.title('Training vs Validation Loss')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.grid(True, alpha=0.3); plt.legend()
plt.tight_layout()
plt.savefig('training_vs_validation_curves.png', dpi=300)
plt.show()

# 3. พล็อตกราฟ Confusion Matrix
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASSES)
disp.plot(ax=ax, cmap='Blues', values_format='d')
plt.title('Float32 Test Set — Confusion Matrix')
plt.savefig('confusion_matrix.png', dpi=300)
plt.show()

print('\n--- Classification Report ---')
print(classification_report(y_true, y_pred, target_names=CLASSES, digits=4, zero_division=0))


## ⚡ ขั้นตอนที่ 6: แปลงโมเดลเป็น Full INT8 Quantization (`.tflite`)
แปลงเป็นโมเดลบีบอัดขนาด **~134 KB** เพื่อรันบนชิป ESP32-S3 โดยเฉพาะ


In [ ]:
# [Cell 7] Full INT8 Quantization & Evaluation
def representative_dataset():
    for imgs, _ in eval_train_ds.unbatch().batch(1).take(150):
        yield [imgs]

print('🔄 กำลังทำการ Full INT8 Quantization...')
converter = tf.lite.TFLiteConverter.from_keras_model(best_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_model_int8 = converter.convert()
tflite_filename = 'mangosteen_separable_cnn_94k_96x96_int8.tflite'

with open(tflite_filename, 'wb') as f:
    f.write(tflite_model_int8)

size_kb = len(tflite_model_int8) / 1024
print(f'🎉 แปลงโมเดลสำเร็จ!')
print(f'📁 ชื่อไฟล์: {tflite_filename}')
print(f'📊 ขนาดไฟล์: {size_kb:.2f} KB (ตรงกับ 134 KB ในเครื่องเป๊ะ)')

# ประเมินผลความแม่นยำโมเดล INT8 บน Test Set
interpreter = tf.lite.Interpreter(model_content=tflite_model_int8)
interpreter.allocate_tensors()
in_det = interpreter.get_input_details()[0]
out_det = interpreter.get_output_details()[0]
in_scale, in_zp = in_det['quantization']

y_pred_int8 = []
for imgs, _ in test_ds:
    img_int8 = np.round(imgs.numpy() / in_scale + in_zp).clip(-128, 127).astype(np.int8)
    interpreter.set_tensor(in_det['index'], img_int8)
    interpreter.invoke()
    y_pred_int8.append(np.argmax(interpreter.get_tensor(out_det['index'])[0]))

int8_acc = np.mean(y_true == np.array(y_pred_int8)) * 100.0
print('=' * 60)
print(f'⚡ INT8 Test Accuracy:    {int8_acc:.2f}% ({np.sum(y_true == np.array(y_pred_int8))}/{len(y_true)} ภาพ)')
print(f'🎯 Float32 Test Accuracy: {test_acc:.2f}%')
print(f'⚖️ Quantization Drop:    {test_acc - int8_acc:+.2f}% (แม่นยำเท่าเดิม!)')
print('=' * 60)


## 💾 ขั้นตอนที่ 7: แปลงเป็น C++ Array และดาวน์โหลดไฟล์ทั้งหมด
แปลงโมเดลเป็น C++ Byte Array พร้อมคำสั่ง `alignas(16)` นำไปใส่โฟลเดอร์ `src/` เพื่อแฟลชลง LilyGO T-SIMCAM


In [ ]:
# [Cell 8] Export C++ Array & Download All Files
prefix = 'mangosteen_model'
header_path = f'{prefix}_data.h'
source_path = f'{prefix}_data.cc'

# 1. สร้าง Header (.h)
with open(header_path, 'w', encoding='utf-8') as f:
    f.write(f'''// Auto-generated Separable CNN Model Header for LilyGO T-SIMCAM (ESP32-S3)
#ifndef {prefix.upper()}_DATA_H_
#define {prefix.upper()}_DATA_H_

#ifdef __cplusplus
extern "C" {{
#endif

extern const unsigned char g_{prefix}_data[];
extern const unsigned int g_{prefix}_data_len;

#ifdef __cplusplus
}}
#endif

#endif
''')

# 2. สร้าง Source (.cc) พร้อม alignas(16)
with open(source_path, 'w', encoding='utf-8') as f:
    f.write(f'// Auto-generated Separable CNN Model Data for LilyGO T-SIMCAM (ESP32-S3)\n')
    f.write(f'#include "{prefix}_data.h"\n\n')
    f.write(f'alignas(16) const unsigned char g_{prefix}_data[] = {{\n')
    for i in range(0, len(tflite_model_int8), 12):
        chunk = tflite_model_int8[i:i+12]
        line = '  ' + ', '.join(f'0x{b:02x}' for b in chunk)
        if i + 12 < len(tflite_model_int8): line += ','
        f.write(line + '\n')
    f.write(f'}};\n\nconst unsigned int g_{prefix}_data_len = {len(tflite_model_int8)};\n')

print(f'✅ สร้าง {header_path} และ {source_path} สำเร็จ!')

# 3. ดาวน์โหลดไฟล์ทั้งหมดลงเครื่อง
print('\n📥 กำลังเริ่มดาวน์โหลดไฟล์ทั้งหมดลงเครื่องของคุณ...')
files.download(tflite_filename)
files.download('mangosteen_separable_cnn_94k.keras')
files.download(header_path)
files.download(source_path)
